# P05 社内データをつなぐ ― 安全に読む

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

顧客台帳・規程・在庫・月次の数字のサーバに MCP の Client で直接つなぎ、資料と道具の違い・利用者ごとの出し分け・監査ログ・キャッシュの目安・数字の照合を確かめます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- API キーが無ければ、講師の実行記録を再生します (費用 0)。記録も無ければ mock で最後まで流れます (mock の数字は性能の証拠になりません)。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. Claude の呼び先を決める (キーは表示しない) と、ラボのモジュールを読めるようにする
from common.nb import choose_provider, safe
from common.kit import use_labs, utf8_console
PROVIDER = choose_provider(IN_COLAB)
use_labs(); utf8_console()

## 1. 資料 (resource) と道具 (tool)
4 つのサーバの資料・資料のひな形・道具を並べ、`crm://policy` と `kb://regulations/a-05` を読みます。どちらにするかの使い分けも最後に出します。

In [ ]:
from run_internal_data import amain as internal
with safe():
    r = await internal(["--only", "resources"])

## 2. 利用者ごとの出し分け
同じ `get_customer` を 9 人の名前で呼びます。法人 C-002 と個人 IND-001 で、だれにどの項目が見えるかを表にします。隠すのはサーバの側 (`common/policy.py`) です。

In [ ]:
with safe():
    r = await internal(["--only", "visibility", "--customer", "C-002,IND-001"])

## 3. 21 社すべてを規程の正解と照合
9 人 × 21 社 = 189 組を、規程から作った正解 (`data/crm/access_gold.json`) と突き合わせます。

In [ ]:
with safe():
    r = await internal(["--only", "visibility", "--all"])

## 4. 監査ログ
照会はサーバの側で `audit.jsonl` に 1 行ずつ残ります。だれが・どの顧客の・どの項目を隠されたか。書き込み先は実行ごとの一時フォルダです。

In [ ]:
with safe():
    r = await internal(["--only", "audit"])

## 5. 一覧のキャッシュの目安
`tools/list` などの結果に付く `ttl_ms` と `cache_scope` を、サーバごとに見ます。読むだけの印 (read_only_hint) の数も並べます。

In [ ]:
with safe():
    r = await internal(["--only", "cache"])

## 6. 読むだけの数字を正解と照合
月次の売上・問い合わせ・在庫割れを report サーバから取り、正解 (`data/report/gold_numbers.json`) と照合します。

In [ ]:
with safe():
    r = await internal(["--only", "report"])